In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, recall_score, precision_score, f1_score, accuracy_score
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMClassifier
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification


: 

In [ ]:
fetal_data = pd.read_csv('ndata.csv')

: 

In [4]:
fetal_data.head()

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


In [5]:
fetal_data.tail()

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
2121,140.0,0.000,0.000,0.007,0.0,0.0,0.0,79.0,0.2,25.0,...,137.0,177.0,4.0,0.0,153.0,150.0,152.0,2.0,0.0,2.0
2122,140.0,0.001,0.000,0.007,0.0,0.0,0.0,78.0,0.4,22.0,...,103.0,169.0,6.0,0.0,152.0,148.0,151.0,3.0,1.0,2.0
2123,140.0,0.001,0.000,0.007,0.0,0.0,0.0,79.0,0.4,20.0,...,103.0,170.0,5.0,0.0,153.0,148.0,152.0,4.0,1.0,2.0
2124,140.0,0.001,0.000,0.006,0.0,0.0,0.0,78.0,0.4,27.0,...,103.0,169.0,6.0,0.0,152.0,147.0,151.0,4.0,1.0,2.0
2125,142.0,0.002,0.002,0.008,0.0,0.0,0.0,74.0,0.4,36.0,...,117.0,159.0,2.0,1.0,145.0,143.0,145.0,1.0,0.0,1.0


In [6]:
fetal_data.shape

(2126, 22)

In [7]:
fetal_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column                                                  Non-Null Count  Dtype  
---  ------                                                  --------------  -----  
 0   baseline value                                          2126 non-null   float64
 1   accelerations                                           2126 non-null   float64
 2   fetal_movement                                          2126 non-null   float64
 3   uterine_contractions                                    2126 non-null   float64
 4   light_decelerations                                     2126 non-null   float64
 5   severe_decelerations                                    2126 non-null   float64
 6   prolongued_decelerations                                2126 non-null   float64
 7   abnormal_short_term_variability                         2126 non-null   float64
 8   mean_value_of_short_term_variability  

In [8]:
fetal_data.isnull().sum()

baseline value                                            0
accelerations                                             0
fetal_movement                                            0
uterine_contractions                                      0
light_decelerations                                       0
severe_decelerations                                      0
prolongued_decelerations                                  0
abnormal_short_term_variability                           0
mean_value_of_short_term_variability                      0
percentage_of_time_with_abnormal_long_term_variability    0
mean_value_of_long_term_variability                       0
histogram_width                                           0
histogram_min                                             0
histogram_max                                             0
histogram_number_of_peaks                                 0
histogram_number_of_zeroes                                0
histogram_mode                          

In [9]:
fetal_data.describe()

,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,...,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
count,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000,2126.00000,...,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000,2126.000000
mean,133.303857,0.003178,0.009481,0.004366,0.001889,0.000003,0.000159,46.990122,1.332785,9.84666,...,93.579492,164.025400,4.068203,0.323612,137.452023,134.610536,138.090310,18.808090,0.320320,1.304327
std,9.840844,0.003866,0.046666,0.002946,0.002960,0.000057,0.000590,17.192814,0.883241,18.39688,...,29.560212,17.944183,2.949386,0.706059,16.381289,15.593596,14.466589,28.977636,0.610829,0.614377
min,106.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,12.000000,0.200000,0.00000,...,50.000000,122.000000,0.000000,0.000000,60.000000,73.000000,77.000000,0.000000,-1.000000,1.000000
25%,126.000000,0.000000,0.000000,0.002000,0.000000,0.000000,0.000000,32.000000,0.700000,0.00000,...,67.000000,152.000000,2.000000,0.000000,129.000000,125.000000,129.000000,2.000000,0.000000,1.000000
50%,133.000000,0.002000,0.000000,0.004000,0.000000,0.000000,0.000000,49.000000,1.200000,0.00000,...,93.000000,162.000000,3.000000,0.000000,139.000000,136.000000,139.000000,7.000000,0.000000,1.000000
75%,140.000000,0.006000,0.003000,0.007000,0.003000,0.000000,0.000000,61.000000,1.700000,11.00000,...,120.000000,174.000000,6.000000,0.000000,148.000000,145.000000,148.000000,24.000000,1.000000,1.000000
max,160.000000,0.019000,0.481000,0.015000,0.015000,0.001000,0.005000,87.000000,7.000000,91.00000,...,159.000000,238.000000,18.000000,10.000000,187.000000,182.000000,186.000000,269.000000,1.000000,3.000000


In [10]:
fetal_data['fetal_health'].value_counts()

fetal_health
1.0    1655
2.0     295
3.0     176
Name: count, dtype: int64

In [11]:
selected_features = ['prolongued_decelerations',
                     'percentage_of_time_with_abnormal_long_term_variability',
                     'accelerations',
                     'abnormal_short_term_variability',
                     'severe_decelerations',
                     'histogram_variance',
                     'light_decelerations',
                     'histogram_min',
                     'uterine_contractions',
                     'mean_value_of_short_term_variability',
                     'histogram_mean',
                     'baseline_value',
                     'fetal_health']

In [12]:
ndata=fetal_data[selected_features]


In [13]:
X = ndata.drop(columns='fetal_health', axis=1)
Y = ndata['fetal_health']

In [44]:
df=pd.DataFrame(ndata)
df.to_csv('ndata.csv', index=False)  

In [14]:
ndata.head()

,prolongued_decelerations,percentage_of_time_with_abnormal_long_term_variability,accelerations,abnormal_short_term_variability,severe_decelerations,histogram_variance,light_decelerations,histogram_min,uterine_contractions,mean_value_of_short_term_variability,histogram_mean,baseline value,fetal_health
0,0.0,43.0,0.000,73.0,0.0,73.0,0.000,62.0,0.000,0.5,137.0,120.0,2.0
1,0.0,0.0,0.006,17.0,0.0,12.0,0.003,68.0,0.006,2.1,136.0,132.0,1.0
2,0.0,0.0,0.003,16.0,0.0,13.0,0.003,68.0,0.008,2.1,135.0,133.0,1.0
3,0.0,0.0,0.003,16.0,0.0,13.0,0.003,53.0,0.008,2.4,134.0,134.0,1.0
4,0.0,0.0,0.007,16.0,0.0,11.0,0.000,53.0,0.008,2.4,136.0,132.0,1.0


In [15]:
print(ndata['fetal_health'])

0       2.0
1       1.0
2       1.0
3       1.0
4       1.0
       ... 
2121    2.0
2122    2.0
2123    2.0
2124    2.0
2125    1.0
Name: fetal_health, Length: 2126, dtype: float64


### Mappling 1.0->0,2.0->1 and 3.0->2

In [16]:
X.shape

(2126, 12)

#### scale

In [17]:
from sklearn.preprocessing import StandardScaler
# Initialize the StandardScaler
scaler = StandardScaler()
# Fit and transform the data
X_scale = scaler.fit_transform(X)

In [18]:
print(Y)

0       2.0
1       1.0
2       1.0
3       1.0
4       1.0
       ... 
2121    2.0
2122    2.0
2123    2.0
2124    2.0
2125    1.0
Name: fetal_health, Length: 2126, dtype: float64


In [19]:
print(X_scale.shape)
print(Y.shape)

(2126, 12)
(2126,)


### sampling using SMOTE

In [21]:
from imblearn.over_sampling import SMOTE
from collections import Counter
# Initialize SMOTE
smote = SMOTE(sampling_strategy='auto', random_state=42)

# Apply SMOTE to generate synthetic samples
X_smote, y_smote = smote.fit_resample(X, Y)

# Count class occurrences after SMOTE
resampled_class_counts = pd.Series(y_smote).value_counts()
print("\nClass sizes after SMOTE:")
print(resampled_class_counts)


Class sizes after SMOTE:
fetal_health
2.0    1655
1.0    1655
3.0    1655
Name: count, dtype: int64


In [22]:
print(y_smote)

0       2.0
1       1.0
2       1.0
3       1.0
4       1.0
       ... 
4960    3.0
4961    3.0
4962    3.0
4963    3.0
4964    3.0
Name: fetal_health, Length: 4965, dtype: float64


In [23]:
print(X_smote.shape)
print(y_smote.shape)

(4965, 12)
(4965,)


In [24]:
mapping = {1.0: 0, 2.0: 1, 3.0: 2}

# Applying the mapping to y
Y_mapped = y_smote.map(mapping)

In [25]:
Y_mapped.shape

(4965,)

In [26]:
X_train, X_test, Y_train, Y_test = train_test_split(X_smote, Y_mapped, test_size=0.1, stratify=Y_mapped, random_state=42)

In [27]:
print("Y_shape=",Y.shape, ";Y_train shape=",Y_train.shape," and Y_test shape=", Y_test.shape)

Y_shape= (2126,) ;Y_train shape= (4468,)  and Y_test shape= (497,)


In [34]:
print("X_shape=",X.shape, ";X_train shape=",X_train.shape," and X_test shape=", X_test.shape)

X_shape= (2126, 12) ;X_train shape= (4468, 12)  and X_test shape= (497, 12)


In [35]:
model = LGBMClassifier()

In [36]:
model.fit(X_train, Y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000553 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2817
[LightGBM] [Info] Number of data points in the train set: 4468, number of used features: 12
[LightGBM] [Info] Start training from score -1.098165
[LightGBM] [Info] Start training from score -1.098836
[LightGBM] [Info] Start training from score -1.098836


LGBMClassifier()

In [38]:
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [39]:
print('Accuracy on Training data : ', training_data_accuracy)

Accuracy on Training data :  0.9997761862130707


In [40]:
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [41]:
print('Accuracy on Test data : ', test_data_accuracy)

Accuracy on Test data :  0.9798792756539235


In [45]:
# input_data = (56,1,1,120,236,0,1,178,0,0.8,2,0,2)
input_data = [0.0, 0.0, 0.005, 27.0, 0.0, 14.0, 0.001, 77.0, 0.008, 1.7, 168.0, 158.0]

In [47]:
input_data_as_numpy_array= np.asarray(input_data)
input_data_reshaped = input_data_as_numpy_array.reshape(1,-1)
prediction = model.predict(input_data_reshaped)
print(prediction)
if (prediction[0]== 0):
    print('Normal')
elif (prediction==1):
    print("suspected")
elif(prediction==2):
    print('Pathologically affected')
else:
    print("unknow")

[0]
Normal
